# Notebook 02 — Historical Replication: CLMX Variance Decomposition

**Project Parallax | Phase 0 / Phase 1 Bridge**

---

## Purpose

This notebook implements and applies the variance decomposition methodology from:

> Campbell, J. Y., Lettau, M., Malkiel, B. G., & Xu, Y. (2001). *Have Individual Stocks Become More Volatile? An Empirical Exploration of Idiosyncratic Risk.* Journal of Finance, 56(1), 1–43.

Updated and extended in:

> Campbell, J. Y., Lettau, M., Malkiel, B. G., & Xu, Y. (2022). *Idiosyncratic Equity Risk Two Decades Later.* NBER Working Paper No. 29916.

The goal is **not** exact numerical replication — the original authors used the full CRSP universe, which requires institutional data access. The goal is:

1. Implement the same methodology as faithfully as public data allows
2. Produce a time series of MKT, IND, and FIRM variance using S&P 500 daily returns
3. Compare the **directional behavior** of our series to the published CLMX 2022 results
4. Document every meaningful deviation from the original study
5. Lay the methodological foundation for extending the analysis into more recent periods

**This notebook does not test H1.** It establishes whether the replication framework behaves consistently with the literature before any Parallax-specific hypothesis testing begins.

---

## Structure

| Section | Topic | Type |
|---|---|---|
| 1 | Mathematical framework | Explanation |
| 2 | Synthetic worked example | Manual calculation |
| 3 | Industry classification pipeline | Data infrastructure |
| 4 | Data loading and preparation | Data retrieval |
| 5 | Variance decomposition — step by step | Core implementation |
| 6 | Monthly and annual aggregation | Aggregation |
| 7 | Results visualization | Analysis |
| 8 | Directional comparison to CLMX 2022 | Validation |
| 9 | Documented deviations and limitations | Methodology |

---

**Build constraint:** This is a learning-first artifact. Each computation is shown manually before it is wrapped in a function. Work through Section 2 by hand before running Section 5.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import io
import zipfile
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from typing import Dict, Optional, Tuple

warnings.filterwarnings('ignore')

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../')
DATA_DIR     = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

# ── Study window (Phase 1 prototype; see docs/methodology.md) ────────────────
# Primary target: 2005–2024. Fallback: 2010–2024 if coverage is too thin.
STUDY_START = '2010-01-01'   # Fallback; update after Phase 1 coverage assessment
STUDY_END   = '2024-12-31'

print(f'Study window: {STUDY_START} → {STUDY_END}')
print(f'Data cache:   {DATA_DIR.resolve()}')

---

## Section 1 — Mathematical Framework

### 1.1 The Return Decomposition

CLMX decompose each stock's daily return into three orthogonal components:

$$r_{i,j,t,d} = \underbrace{\mu_{t,d}}_{\text{market}} + \underbrace{\eta_{j,t,d}}_{\text{industry excess market}} + \underbrace{\varepsilon_{i,j,t,d}}_{\text{firm excess industry}}$$

Where:
- $r_{i,j,t,d}$ = return of stock $i$ in industry $j$ on day $d$ of month $t$
- $\mu_{t,d}$ = value-weighted market return on day $d$
- $\eta_{j,t,d} = r_{j,t,d} - \mu_{t,d}$ = value-weighted industry $j$ return **minus** market return
- $\varepsilon_{i,j,t,d} = r_{i,t,d} - r_{j,t,d}$ = firm $i$ return **minus** its industry return

The three components are **orthogonal by construction** — because each is defined as an excess over the prior level, they do not double-count.

### 1.2 Monthly Variance Accumulation

For each calendar month $t$, CLMX accumulate squared daily components:

$$\text{MKT}_t = \sum_{d=1}^{D_t} \mu_{t,d}^2$$

$$\text{IND}_t = \sum_j W_{j,t} \cdot \sum_{d=1}^{D_t} \eta_{j,t,d}^2$$

$$\text{FIRM}_t = \sum_j W_{j,t} \cdot \sum_{i \in j} w_{i,j,t} \cdot \sum_{d=1}^{D_t} \varepsilon_{i,j,t,d}^2$$

Where:
- $D_t$ = number of trading days in month $t$
- $W_{j,t}$ = market-cap weight of industry $j$ at start of month $t$
- $w_{i,j,t}$ = weight of firm $i$ **within** industry $j$ at start of month $t$

**Key design choices confirmed by the CLMX authors' own 2022 restatement (NBER WP 29916):**

1. **Raw squared returns** — daily returns are NOT demeaned before squaring. The components are already defined as excess returns, so no additional mean-subtraction is needed.
2. **Not normalized by day count** — the monthly sum is left as a raw accumulation, not divided by $D_t$. Months with more trading days will have higher raw variance (a feature, not a bug — it reflects more price movement in longer months).
3. **Value-weighted primary** — both VW and EW versions are reported; VW is the base case.

### 1.3 Annual Aggregation

Annual variance estimates are obtained by summing the 12 (or however many available) monthly variance estimates:

$$\text{MKT}_{\text{year}} = \sum_{t \in \text{year}} \text{MKT}_t$$

Annual **volatility** (standard deviation) = $\sqrt{\text{Annual Variance}}$, which puts the series in the same units as annualized return standard deviation.

### 1.4 What Each Component Measures

| Component | What it captures | Expected behavior |
|---|---|---|
| MKT | Variance common to all stocks (systematic market risk) | Spikes during crisis periods |
| IND | Variance common within industries but orthogonal to the market | Sector rotation and industry-level shocks |
| FIRM | Variance specific to individual companies after market and industry are removed | The "idiosyncratic" component |

CLMX's 2001 finding: FIRM increased secularly over 1962–1997. The 2022 update found this trend reversed post-2001, with market volatility rising more than firm volatility in the post-crisis period.

---

## Section 2 — Synthetic Worked Example

Before using real data, we verify the decomposition on a small synthetic case where we know what the answer should be.

**Setup:** 3 stocks, 2 industries, 5 trading days.
- Industry A (Technology): stocks TECH_1, TECH_2
- Industry B (Energy): stock ENRG_1

Work through the decomposition by hand alongside the code.

In [ ]:
# ── Synthetic data setup ──────────────────────────────────────────────────────
# Daily returns for 3 stocks over 5 trading days
synth_returns = pd.DataFrame({
    'TECH_1': [ 0.020,  0.010, -0.005,  0.015,  0.003],
    'TECH_2': [ 0.012,  0.008, -0.010,  0.020, -0.002],
    'ENRG_1': [-0.005,  0.003,  0.008, -0.003,  0.010],
})

# Industry membership
industry_map = {'TECH_1': 'Technology', 'TECH_2': 'Technology', 'ENRG_1': 'Energy'}

# Market-cap weights (beginning of month; these are total market weights, not within-industry)
# TECH_1: 40% of market, TECH_2: 30%, ENRG_1: 30%
mkt_weights = pd.Series({'TECH_1': 0.40, 'TECH_2': 0.30, 'ENRG_1': 0.30})

print('Daily returns:')
print(synth_returns)
print(f'\nMarket weights: {mkt_weights.to_dict()}')
print(f'Industry map:   {industry_map}')

In [ ]:
# ── Step 2a: Value-weighted market return each day ────────────────────────────
# μ_d = Σ_i w_i * r_i,d
mu = synth_returns.mul(mkt_weights, axis='columns').sum(axis='columns')
print('Daily market return (μ_d):')
print(mu.round(6))
print(f'\nMKT monthly variance = Σ μ_d² = {(mu**2).sum():.8f}')

In [ ]:
# ── Step 2b: Value-weighted industry returns each day ─────────────────────────
# r_j,d = VW average return of stocks in industry j on day d
# Within-industry weights: each stock's market weight, normalized to sum to 1 within industry

industry_groups = {}
for industry in set(industry_map.values()):
    members = [t for t, ind in industry_map.items() if ind == industry]
    ind_mkt_weights = mkt_weights[members]
    within_weights = ind_mkt_weights / ind_mkt_weights.sum()   # normalize within industry
    industry_groups[industry] = {
        'members': members,
        'within_weights': within_weights,
        'industry_mkt_weight': ind_mkt_weights.sum(),           # W_j = industry share of total market
    }
    r_j = synth_returns[members].mul(within_weights, axis='columns').sum(axis='columns')
    industry_groups[industry]['r_j'] = r_j

for ind, data in industry_groups.items():
    print(f'\n{ind}:')
    print(f'  Members:            {data["members"]}')
    print(f'  Within-ind weights: {data["within_weights"].to_dict()}')
    print(f'  Industry mkt weight (W_j): {data["industry_mkt_weight"]:.2f}')
    print(f'  VW industry return r_j,d: {data["r_j"].round(6).to_dict()}')

In [ ]:
# ── Step 2c: Industry excess-market component (η) ────────────────────────────
# η_j,d = r_j,d - μ_d

for ind, data in industry_groups.items():
    eta = data['r_j'] - mu
    industry_groups[ind]['eta'] = eta
    print(f'{ind} η_j,d (industry excess market): {eta.round(6).to_dict()}')

# IND monthly variance = Σ_j W_j * Σ_d η_j,d²
IND_synth = sum(
    data['industry_mkt_weight'] * (data['eta']**2).sum()
    for data in industry_groups.values()
)
print(f'\nIND monthly variance = Σ_j W_j * Σ_d η²_j,d = {IND_synth:.8f}')

In [ ]:
# ── Step 2d: Firm excess-industry component (ε) and FIRM variance ─────────────
# ε_i,d = r_i,d - r_j,d

FIRM_synth = 0.0

for ind, data in industry_groups.items():
    print(f'\n{ind} (W_j = {data["industry_mkt_weight"]:.2f}):')
    ind_firm_contribution = 0.0
    for ticker in data['members']:
        within_w = data['within_weights'][ticker]
        epsilon = synth_returns[ticker] - data['r_j']
        firm_var = (epsilon**2).sum()
        contribution = data['industry_mkt_weight'] * within_w * firm_var
        FIRM_synth += contribution
        ind_firm_contribution += contribution
        print(f'  {ticker}: w_i|j={within_w:.3f}, ε: {epsilon.round(6).to_dict()}')
        print(f'  → Σ ε² = {firm_var:.8f}, contribution = {contribution:.8f}')

print(f'\nFIRM monthly variance = {FIRM_synth:.8f}')

In [ ]:
# ── Step 2e: Reconciliation ───────────────────────────────────────────────────
# The three components should approximately equal the VW-average individual stock variance.
# They are not guaranteed to sum exactly to the VW-total (cross-products from the
# decomposition can be non-zero in finite samples), but should be very close.

MKT_synth = (mu**2).sum()

# VW average of individual stock monthly variances
stock_monthly_var = (synth_returns**2).sum()   # Σ_d r_i,d² for each stock
VW_total = (stock_monthly_var * mkt_weights).sum()

# Sum of components
components_sum = MKT_synth + IND_synth + FIRM_synth

print('═' * 55)
print('Synthetic Worked Example — Reconciliation')
print('═' * 55)
print(f'MKT                             : {MKT_synth:.8f}')
print(f'IND                             : {IND_synth:.8f}')
print(f'FIRM                            : {FIRM_synth:.8f}')
print(f'─────────────────────────────────────────────────────')
print(f'Sum of components               : {components_sum:.8f}')
print(f'VW-avg individual stock variance: {VW_total:.8f}')
print(f'Difference (cross-product terms): {VW_total - components_sum:.2e}')
print()
print('Variance shares:')
print(f'  MKT  share: {MKT_synth / components_sum:.1%}')
print(f'  IND  share: {IND_synth / components_sum:.1%}')
print(f'  FIRM share: {FIRM_synth / components_sum:.1%}')

**Check before proceeding:** The three components should sum to approximately the VW-average individual stock variance. Small differences are expected (these are the cross-product terms from the decomposition identity). If the difference is large (more than 1% of total), revisit the weight calculations.

If this cell produces sensible output — FIRM variance meaningfully positive, IND variance reflecting industry-specific movement, MKT variance reflecting common market movement — the implementation logic is correct. Scale up to real data in Section 5.

---

## Section 3 — Industry Classification Pipeline

The CLMX decomposition requires classifying each stock into one of the **Fama-French 49 industries** using its SIC code.

**Pipeline:**
1. Fetch the FF49 SIC crosswalk from the Kenneth French Data Library
2. Get SIC codes for our S&P 500 tickers from SEC EDGAR
3. Map each ticker → SIC code → FF49 industry

**Why not GICS?** GICS (the S&P/MSCI sector classification available via yfinance metadata) differs structurally from Fama-French industries. Using GICS would make the decomposition non-comparable to CLMX. The Fama-French classification must be used for a faithful methodology comparison.

In [ ]:
# ── 3a: Download and parse the FF49 SIC crosswalk ────────────────────────────
# Source: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
# File: Siccodes49.zip (text file mapping SIC code ranges to FF49 industries)

FF49_CACHE = DATA_DIR / 'ff49_sic_map.csv'

def fetch_ff49_crosswalk() -> pd.DataFrame:
    """Download and parse the Fama-French 49-industry SIC code crosswalk.
    
    Returns a DataFrame with columns:
        sic_lo (int)  : lower bound of SIC range
        sic_hi (int)  : upper bound of SIC range (inclusive)
        industry_num  : FF49 industry number (1–49)
        industry_name : FF49 industry name abbreviation
    """
    if FF49_CACHE.exists():
        print(f'Loading FF49 crosswalk from cache: {FF49_CACHE}')
        return pd.read_csv(FF49_CACHE)

    url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Siccodes49.zip'
    print(f'Fetching FF49 SIC crosswalk from French Data Library...')
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        fname = zf.namelist()[0]
        raw = zf.read(fname).decode('latin-1')

    rows = []
    current_ind_num  = None
    current_ind_name = None

    for line in raw.splitlines():
        line = line.rstrip()
        if not line:
            continue
        # Industry header lines look like: " 1 Agric  Agriculture"
        # SIC range lines look like:       "0100 0199"
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit() and parts[1].isalpha() and len(parts[0]) <= 2:
            current_ind_num  = int(parts[0])
            current_ind_name = parts[1]
        elif len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
            try:
                sic_lo = int(parts[0])
                sic_hi = int(parts[1])
                if current_ind_num is not None:
                    rows.append({
                        'sic_lo': sic_lo,
                        'sic_hi': sic_hi,
                        'industry_num': current_ind_num,
                        'industry_name': current_ind_name,
                    })
            except ValueError:
                pass

    df = pd.DataFrame(rows)
    df.to_csv(FF49_CACHE, index=False)
    print(f'FF49 crosswalk saved to {FF49_CACHE} ({len(df)} SIC ranges across {df["industry_num"].nunique()} industries)')
    return df


def sic_to_ff49(sic: int, crosswalk: pd.DataFrame) -> Tuple[int, str]:
    """Map a 4-digit SIC code to a Fama-French 49 industry.
    
    Returns (industry_num, industry_name). Returns (49, 'Other') for unmatched codes.
    Industry 49 ("Other") is the CLMX convention for unclassified stocks.
    """
    mask = (crosswalk['sic_lo'] <= sic) & (sic <= crosswalk['sic_hi'])
    matches = crosswalk[mask]
    if matches.empty:
        return (49, 'Other')
    row = matches.iloc[0]
    return (int(row['industry_num']), row['industry_name'])


# Download the crosswalk
ff49 = fetch_ff49_crosswalk()
print(f'\nFF49 industries found: {ff49["industry_num"].nunique()}')
print(ff49.head(10))

In [ ]:
# ── 3b: Retrieve SIC codes from SEC EDGAR ────────────────────────────────────
# EDGAR provides SIC codes for all US-registered issuers via a public API.
# Process:
#   1. Download the EDGAR company ticker index → maps tickers to CIK numbers
#   2. For each CIK, fetch the company submission record → includes SIC code
#
# Rate limit: EDGAR asks for max 10 requests/sec. We use a 0.12s delay.

EDGAR_CACHE = DATA_DIR / 'sp500_sic_codes.csv'

def get_sp500_tickers_from_wikipedia() -> list:
    """Get current S&P 500 tickers from Wikipedia."""
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url)
    sp500 = tables[0]
    tickers = sp500['Symbol'].str.replace('.', '-', regex=False).tolist()
    print(f'S&P 500 constituent list retrieved: {pd.Timestamp.now().date()} ({len(tickers)} tickers)')
    return tickers


def fetch_edgar_sic_codes(tickers: list) -> pd.DataFrame:
    """Fetch SIC codes for a list of tickers from SEC EDGAR.
    
    Caches results to avoid repeated API calls.
    EDGAR user-agent policy requires a contact email in the header.
    """
    if EDGAR_CACHE.exists():
        print(f'Loading SIC codes from cache: {EDGAR_CACHE}')
        return pd.read_csv(EDGAR_CACHE, dtype={'sic': str})

    # Step 1: Get EDGAR ticker → CIK mapping
    headers = {'User-Agent': 'Project Parallax research mattnolan.archive@gmail.com'}
    print('Fetching EDGAR ticker-CIK index...')
    r = requests.get('https://www.sec.gov/files/company_tickers.json', headers=headers)
    r.raise_for_status()
    cik_data = r.json()

    # Build ticker → CIK lookup (CIK is zero-padded to 10 digits for API calls)
    ticker_to_cik = {
        v['ticker'].upper(): str(v['cik_str']).zfill(10)
        for v in cik_data.values()
    }
    print(f'EDGAR index loaded: {len(ticker_to_cik)} tickers mapped to CIK numbers')

    # Step 2: Fetch SIC code for each ticker via company submissions endpoint
    records = []
    not_found = []

    for i, ticker in enumerate(tickers):
        ticker_upper = ticker.upper().replace('-', '.')
        # Try both ticker format variants
        cik = ticker_to_cik.get(ticker_upper) or ticker_to_cik.get(ticker.upper())

        if cik is None:
            not_found.append(ticker)
            records.append({'ticker': ticker, 'cik': None, 'sic': None, 'company_name': None})
            continue

        url = f'https://data.sec.gov/submissions/CIK{cik}.json'
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            resp.raise_for_status()
            company_data = resp.json()
            records.append({
                'ticker': ticker,
                'cik': cik,
                'sic': company_data.get('sic'),
                'company_name': company_data.get('name'),
            })
        except Exception as e:
            records.append({'ticker': ticker, 'cik': cik, 'sic': None, 'company_name': None})

        time.sleep(0.12)   # EDGAR rate limit: ~8 req/sec

        if (i + 1) % 50 == 0:
            print(f'  Progress: {i+1}/{len(tickers)} tickers processed')

    df = pd.DataFrame(records)
    df.to_csv(EDGAR_CACHE, index=False)
    print(f'\nSIC codes retrieved: {df["sic"].notna().sum()}/{len(df)} tickers matched')
    if not_found:
        print(f'Not found in EDGAR index: {not_found[:10]}... ({len(not_found)} total)')
    return df


# Retrieve tickers and SIC codes
sp500_tickers = get_sp500_tickers_from_wikipedia()
sic_df = fetch_edgar_sic_codes(sp500_tickers)
print(f'\nSample:')
print(sic_df.head())

In [ ]:
# ── 3c: Map SIC codes to FF49 industries ─────────────────────────────────────

def build_ticker_industry_map(sic_df: pd.DataFrame, ff49: pd.DataFrame) -> pd.DataFrame:
    """Map each ticker to its Fama-French 49 industry using its SIC code."""
    results = []
    for _, row in sic_df.iterrows():
        if pd.isna(row['sic']):
            ind_num, ind_name = 49, 'Other'
        else:
            try:
                sic_int = int(str(row['sic']).replace('.0', ''))
                ind_num, ind_name = sic_to_ff49(sic_int, ff49)
            except (ValueError, TypeError):
                ind_num, ind_name = 49, 'Other'
        results.append({
            'ticker': row['ticker'],
            'sic': row['sic'],
            'industry_num': ind_num,
            'industry_name': ind_name,
        })
    return pd.DataFrame(results).set_index('ticker')


ticker_industry = build_ticker_industry_map(sic_df, ff49)

print('Industry assignment sample:')
print(ticker_industry.head(10))
print(f'\nTotal tickers classified: {len(ticker_industry)}')
print(f'Unique FF49 industries represented: {ticker_industry["industry_num"].nunique()}')
print(f'Tickers assigned to "Other" (industry 49): {(ticker_industry["industry_num"] == 49).sum()}')

print('\nIndustry composition (top 15 by count):')
print(
    ticker_industry.groupby(['industry_num', 'industry_name'])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .to_string()
)

---

## Section 4 — Data Loading and Preparation

In [ ]:
# ── 4a: Fetch daily prices ────────────────────────────────────────────────────
# We download adjusted closing prices and compute daily log returns.
# Using adjusted prices incorporates splits and dividends, matching
# the total-return convention used in CLMX (CRSP total return series).
#
# Cache to disk: yfinance downloads are slow for 500 tickers × 15 years.

PRICES_CACHE = DATA_DIR / 'sp500_daily_prices.parquet'

def fetch_daily_prices(tickers: list, start: str, end: str) -> pd.DataFrame:
    """Download adjusted closing prices for a list of tickers.
    
    Returns a DataFrame with dates as index and tickers as columns.
    Caches to disk to avoid repeated downloads.
    """
    if PRICES_CACHE.exists():
        print(f'Loading prices from cache: {PRICES_CACHE}')
        return pd.read_parquet(PRICES_CACHE)

    print(f'Fetching daily adjusted prices for {len(tickers)} tickers ({start} → {end})...')
    print('This may take several minutes for the full S&P 500 universe.')

    prices = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
    )['Close']

    prices.to_parquet(PRICES_CACHE)
    print(f'Prices saved to {PRICES_CACHE}: {prices.shape}')
    return prices


prices = fetch_daily_prices(sp500_tickers, STUDY_START, STUDY_END)
print(f'\nPrice data shape: {prices.shape} (dates × tickers)')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Missing rate: {prices.isna().mean().mean():.1%} of cells')

In [ ]:
# ── 4b: Compute daily simple returns ─────────────────────────────────────────
# CLMX use simple (not log) returns.
# pct_change() computes (P_t - P_{t-1}) / P_{t-1}.

daily_returns = prices.pct_change().iloc[1:]   # drop the first NaN row

# Filter to tickers that have industry assignments
classified_tickers = list(ticker_industry.index)
available_tickers  = [t for t in classified_tickers if t in daily_returns.columns]
daily_returns = daily_returns[available_tickers]

print(f'Daily returns shape: {daily_returns.shape}')
print(f'Tickers with industry assignments and price data: {len(available_tickers)}')

# Drop (not impute) missing values — following CLMX convention
# Each computation will handle missing data by working with whatever is available
print(f'Missing return cells: {daily_returns.isna().sum().sum()} of {daily_returns.size}')

In [ ]:
# ── 4c: Market capitalization proxy for value weighting ───────────────────────
# CLMX use prior-month market-cap weights updated monthly.
#
# LIMITATION: We do not have daily historical market cap for each stock.
# yfinance provides current market cap via info['marketCap'], which is not
# suitable for historical analysis.
#
# APPROXIMATION: We use daily price × (current shares outstanding) as a
# proportional market-cap proxy. Shares outstanding change slowly relative
# to daily price movements, so this approximation is reasonable for weight
# computation. It will understate market-cap changes from significant share
# issuance or buyback events.
#
# This is a documented deviation from CLMX. The effect is expected to be
# small because weight changes between monthly rebalancings are dominated
# by price changes, not shares-outstanding changes, for most months.

SHARES_CACHE = DATA_DIR / 'sp500_shares_outstanding.csv'

def fetch_shares_outstanding(tickers: list) -> pd.Series:
    """Fetch current shares outstanding from yfinance.
    
    Uses current shares as a static approximation for historical weighting.
    Documented limitation: does not capture historical share-count changes.
    """
    if SHARES_CACHE.exists():
        return pd.read_csv(SHARES_CACHE, index_col=0, squeeze=False).iloc[:, 0]

    print(f'Fetching shares outstanding for {len(tickers)} tickers...')
    shares = {}
    for i, ticker in enumerate(tickers):
        try:
            info = yf.Ticker(ticker).fast_info
            shares[ticker] = getattr(info, 'shares', None)
        except Exception:
            shares[ticker] = None
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(tickers)}')
        time.sleep(0.05)

    s = pd.Series(shares, name='shares_outstanding')
    s.to_csv(SHARES_CACHE)
    print(f'Shares outstanding retrieved: {s.notna().sum()}/{len(s)} tickers')
    return s


shares = fetch_shares_outstanding(available_tickers)

# Compute approximate daily market cap = price × shares
# For tickers with missing shares, use equal weight as fallback (documented)
shares_clean = shares.reindex(available_tickers).fillna(shares.median())
approx_mktcap = prices[available_tickers].mul(shares_clean, axis='columns')

print(f'\nApproximate market cap computed for {approx_mktcap.columns.notna().sum()} tickers')
print(f'Tickers with shares fallback (median): {shares.reindex(available_tickers).isna().sum()}')
print('\nLIMITATION NOTE: Shares outstanding are current values used as historical approximation.')

---

## Section 5 — Variance Decomposition: Step by Step

This section implements the CLMX decomposition for a single example month, then wraps the logic for application to the full study period.

Work through the single-month implementation manually before running the full-period loop.

In [ ]:
# ── 5a: Helper — compute monthly value weights from prior-month market cap ────

def compute_monthly_weights(approx_mktcap: pd.DataFrame, year: int, month: int) -> pd.Series:
    """Compute beginning-of-month market-cap weights from prior month's closing prices.
    
    Returns a Series of weights that sum to 1.0 across the universe.
    Stocks with no data in the prior month receive zero weight.
    """
    # Use the last available market cap from the prior calendar month
    prior_month_end = pd.Timestamp(year=year, month=month, day=1) - pd.offsets.MonthBegin(1)
    prior_month_data = approx_mktcap[
        (approx_mktcap.index >= prior_month_end - pd.offsets.MonthEnd(1)) &
        (approx_mktcap.index <  pd.Timestamp(year=year, month=month, day=1))
    ]

    if prior_month_data.empty:
        # For the first available month, use current month market cap
        cur_month_data = approx_mktcap[
            (approx_mktcap.index.year == year) & (approx_mktcap.index.month == month)
        ]
        last_mktcap = cur_month_data.iloc[0] if not cur_month_data.empty else pd.Series(dtype=float)
    else:
        last_mktcap = prior_month_data.iloc[-1]   # last trading day of prior month

    total = last_mktcap.sum()
    if total == 0:
        return pd.Series(0.0, index=last_mktcap.index)
    return last_mktcap / total


# Test on a single month
test_weights = compute_monthly_weights(approx_mktcap, 2020, 1)
print('January 2020 market-cap weights (top 10 by weight):')
print(test_weights.sort_values(ascending=False).head(10).apply(lambda x: f'{x:.4%}'))

In [ ]:
# ── 5b: Decompose a single month manually ────────────────────────────────────
# March 2020 is chosen as the worked example: large COVID shock, high volatility.
# This month should produce elevated MKT and industry components.

EXAMPLE_YEAR, EXAMPLE_MONTH = 2020, 3

# Select daily returns for March 2020
month_mask = (
    (daily_returns.index.year  == EXAMPLE_YEAR) &
    (daily_returns.index.month == EXAMPLE_MONTH)
)
R = daily_returns[month_mask].copy()
print(f'March 2020: {len(R)} trading days, {R.shape[1]} tickers')

# Market-cap weights from prior month
W = compute_monthly_weights(approx_mktcap, EXAMPLE_YEAR, EXAMPLE_MONTH)
W = W.reindex(R.columns).fillna(0)
W = W / W.sum()   # re-normalize after reindexing to available tickers

# Drop tickers with any missing daily returns in this month
# (CLMX convention: missing returns dropped, not imputed)
valid_cols = R.columns[R.notna().all()]
R = R[valid_cols]
W = W[valid_cols]
W = W / W.sum()   # re-normalize again after dropping missing-return tickers
print(f'Tickers with complete return history for March 2020: {len(valid_cols)}')

In [ ]:
# ── Step 1: Daily value-weighted market return ────────────────────────────────
mu_d = R.mul(W, axis='columns').sum(axis='columns')
MKT_month = (mu_d**2).sum()

print('Daily VW market return, March 2020:')
print(mu_d.round(4).to_string())
print(f'\nMKT monthly variance: {MKT_month:.6f}')
print(f'Annualized market volatility (×12): {np.sqrt(MKT_month * 12):.2%}')

In [ ]:
# ── Steps 2–4: Industry and firm components ───────────────────────────────────

IND_month  = 0.0
FIRM_month = 0.0

industry_detail = {}   # for inspection

for ind_num in ticker_industry.loc[valid_cols, 'industry_num'].unique():
    members_all = ticker_industry[
        ticker_industry['industry_num'] == ind_num
    ].index.tolist()
    members = [m for m in members_all if m in R.columns]   # available in this month
    if not members:
        continue

    # Industry weight in the total market (sum of member market-cap weights)
    W_j = W[members].sum()

    # Within-industry weights (normalized to sum to 1 within the industry)
    w_ij = W[members] / W_j

    # Value-weighted industry return each day
    r_j = R[members].mul(w_ij, axis='columns').sum(axis='columns')

    # Industry excess-market component (η)
    eta_j = r_j - mu_d

    # IND component from this industry
    ind_contribution = W_j * (eta_j**2).sum()
    IND_month += ind_contribution

    # Firm residuals (ε) for each stock in this industry
    firm_contribution = 0.0
    for ticker in members:
        eps_i = R[ticker] - r_j
        firm_contribution += W_j * w_ij[ticker] * (eps_i**2).sum()
    FIRM_month += firm_contribution

    ind_name = ticker_industry.loc[members[0], 'industry_name']
    industry_detail[ind_num] = {
        'name': ind_name, 'members': len(members),
        'W_j': W_j, 'IND': ind_contribution, 'FIRM': firm_contribution,
    }

# ── Results for March 2020 ────────────────────────────────────────────────────
total_var = MKT_month + IND_month + FIRM_month
print('═' * 60)
print(f'CLMX Variance Decomposition — March 2020')
print('═' * 60)
print(f'MKT  monthly variance : {MKT_month:.6f}  ({MKT_month/total_var:.1%} of total)')
print(f'IND  monthly variance : {IND_month:.6f}  ({IND_month/total_var:.1%} of total)')
print(f'FIRM monthly variance : {FIRM_month:.6f}  ({FIRM_month/total_var:.1%} of total)')
print(f'─────────────────────────────────────────────────────────────')
print(f'Sum of components     : {total_var:.6f}')
print()
print('Interpretation: In a crisis month (COVID crash), the market component should')
print('dominate — all stocks fell together. A high MKT share relative to FIRM is expected.')

In [ ]:
# ── 5c: Top industries by IND and FIRM contribution in March 2020 ─────────────
ind_detail_df = pd.DataFrame(industry_detail).T.sort_values('W_j', ascending=False)
print('Industry breakdown (top 15 by market weight):')
print(
    ind_detail_df.head(15)[['name', 'members', 'W_j', 'IND', 'FIRM']]
    .assign(
        W_j  = lambda x: x['W_j'].map('{:.2%}'.format),
        IND  = lambda x: x['IND'].map('{:.6f}'.format),
        FIRM = lambda x: x['FIRM'].map('{:.6f}'.format),
    )
    .to_string()
)

---

## Section 6 — Full-Period Decomposition

Apply the monthly decomposition across the full study window. This wraps the Section 5 logic into a function and loops over every month.

In [ ]:
# ── 6a: Wrap the monthly decomposition ───────────────────────────────────────

def clmx_decompose_month(
    daily_returns: pd.DataFrame,
    approx_mktcap: pd.DataFrame,
    ticker_industry: pd.DataFrame,
    year: int,
    month: int,
    min_valid_days: int = 10,
) -> Optional[Dict]:
    """Compute CLMX variance decomposition for a single calendar month.

    Parameters
    ----------
    daily_returns  : Daily simple returns (dates × tickers)
    approx_mktcap  : Approximate daily market cap (dates × tickers)
    ticker_industry: Ticker → FF49 industry mapping (from build_ticker_industry_map)
    year, month    : Calendar period to decompose
    min_valid_days : Minimum trading days a stock must have to be included.
                     Not an original CLMX requirement; documented as a design choice.

    Returns
    -------
    dict with keys: year, month, MKT, IND, FIRM, n_stocks, n_industries, n_trading_days
    Returns None if the month has no data.
    """
    mask = (
        (daily_returns.index.year  == year) &
        (daily_returns.index.month == month)
    )
    R = daily_returns[mask].copy()
    if len(R) < min_valid_days:
        return None

    # Drop tickers with any missing returns in this month
    valid_cols = R.columns[R.notna().all()]
    R = R[valid_cols]
    if R.empty:
        return None

    # Market-cap weights from prior month
    W = compute_monthly_weights(approx_mktcap, year, month)
    W = W.reindex(valid_cols).fillna(0)
    total_w = W.sum()
    if total_w == 0:
        return None
    W = W / total_w

    # Daily VW market return and MKT variance
    mu_d = R.mul(W, axis='columns').sum(axis='columns')
    MKT  = (mu_d**2).sum()

    IND  = 0.0
    FIRM = 0.0
    industries_seen = set()

    for ind_num in ticker_industry.loc[valid_cols, 'industry_num'].unique():
        members = [
            t for t in ticker_industry[
                ticker_industry['industry_num'] == ind_num
            ].index
            if t in R.columns
        ]
        if not members:
            continue

        W_j  = W[members].sum()
        if W_j == 0:
            continue
        w_ij = W[members] / W_j
        r_j  = R[members].mul(w_ij, axis='columns').sum(axis='columns')
        eta  = r_j - mu_d
        IND += W_j * (eta**2).sum()

        for ticker in members:
            eps = R[ticker] - r_j
            FIRM += W_j * w_ij[ticker] * (eps**2).sum()

        industries_seen.add(ind_num)

    return {
        'year': year, 'month': month,
        'MKT': MKT, 'IND': IND, 'FIRM': FIRM,
        'n_stocks': len(valid_cols),
        'n_industries': len(industries_seen),
        'n_trading_days': len(R),
    }

In [ ]:
# ── 6b: Run decomposition across the full study window ────────────────────────
# This is the core computational loop. It will take several minutes for
# a 15-year study window with 500 tickers.

RESULTS_CACHE = DATA_DIR / 'clmx_monthly_decomposition.csv'

if RESULTS_CACHE.exists():
    print(f'Loading decomposition results from cache: {RESULTS_CACHE}')
    monthly_results = pd.read_csv(RESULTS_CACHE)
else:
    print('Running monthly variance decomposition...')
    records = []

    study_months = pd.date_range(start=STUDY_START, end=STUDY_END, freq='MS')
    for dt in study_months:
        result = clmx_decompose_month(
            daily_returns, approx_mktcap, ticker_industry,
            year=dt.year, month=dt.month,
        )
        if result is not None:
            records.append(result)
            if dt.month == 1 or dt == study_months[-1]:
                print(f'  Completed through {dt.year}-{dt.month:02d}')

    monthly_results = pd.DataFrame(records)
    monthly_results.to_csv(RESULTS_CACHE, index=False)
    print(f'\nDecomposition complete: {len(monthly_results)} months saved to {RESULTS_CACHE}')

monthly_results['date'] = pd.to_datetime(
    monthly_results[['year', 'month']].assign(day=1)
)
monthly_results = monthly_results.set_index('date').sort_index()

print(f'\nMonthly results: {len(monthly_results)} months')
print(monthly_results.head())

In [ ]:
# ── 6c: Annual aggregation ────────────────────────────────────────────────────
# Annual variance = sum of monthly variances
# Annual volatility = sqrt(annual variance) — same units as annualized return std dev

annual = monthly_results[['MKT', 'IND', 'FIRM']].resample('YE').sum()
annual.index = annual.index.year

# Annualized volatility (standard deviation)
annual_vol = np.sqrt(annual)
annual_vol.columns = ['MKT_vol', 'IND_vol', 'FIRM_vol']

# Variance shares
annual['total'] = annual['MKT'] + annual['IND'] + annual['FIRM']
annual['MKT_share']  = annual['MKT']  / annual['total']
annual['IND_share']  = annual['IND']  / annual['total']
annual['FIRM_share'] = annual['FIRM'] / annual['total']

print('Annual variance decomposition:')
print(annual[['MKT_share', 'IND_share', 'FIRM_share']].round(3).to_string())

---

## Section 7 — Results Visualization

In [ ]:
# ── 7a: Time series of annualized volatility components ───────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    'CLMX Variance Decomposition — S&P 500 (Public Data Approximation)\n'
    'Annualized Volatility by Component',
    fontsize=13, fontweight='bold'
)

colors = {'MKT': '#2166ac', 'IND': '#4dac26', 'FIRM': '#d01c8b'}
labels = {'MKT': 'Market', 'IND': 'Industry', 'FIRM': 'Firm-Specific'}

for ax, component in zip(axes, ['MKT', 'IND', 'FIRM']):
    # 12-month rolling sum of monthly variance → annualized vol
    rolling_vol = np.sqrt(monthly_results[component].rolling(12).sum())
    ax.plot(rolling_vol.index, rolling_vol * 100,
            color=colors[component], linewidth=1.5, label='12m rolling')
    ax.fill_between(rolling_vol.index, rolling_vol * 100, alpha=0.15,
                    color=colors[component])
    ax.set_ylabel(f'{labels[component]}\nVol (% ann.)', fontsize=10)
    ax.grid(alpha=0.3, linestyle='--')
    ax.legend(loc='upper right', fontsize=9)

# Mark key economic events
events = [
    ('2008-09-15', 'GFC'),
    ('2020-03-01', 'COVID'),
    ('2022-01-01', 'Rate hikes'),
]
for date_str, label in events:
    for ax in axes:
        ax.axvline(pd.Timestamp(date_str), color='gray', linestyle=':', alpha=0.6, linewidth=0.8)
        ax.text(pd.Timestamp(date_str), ax.get_ylim()[1] * 0.95, label,
                fontsize=7, color='gray', ha='center', va='top')

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_01_clmx_components_time_series.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# ── 7b: Variance share chart — the primary H1 visualization ──────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# 24-month rolling variance shares
rolling_total = monthly_results[['MKT', 'IND', 'FIRM']].rolling(24).sum()
rolling_shares = rolling_total.div(rolling_total.sum(axis=1), axis=0)

ax.stackplot(
    rolling_shares.index,
    rolling_shares['FIRM'] * 100,
    rolling_shares['IND'] * 100,
    rolling_shares['MKT'] * 100,
    labels=['Firm-Specific', 'Industry', 'Market'],
    colors=['#d01c8b', '#4dac26', '#2166ac'],
    alpha=0.8,
)

ax.set_ylabel('Variance Share (%)')
ax.set_xlabel('Date')
ax.set_title(
    'Variance Shares: Market / Industry / Firm — 24-Month Rolling\n'
    'S&P 500 (Public Data Approximation) | Compare to CLMX (2022) Figure 4'
)
ax.legend(loc='upper left', fontsize=10)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3, linestyle='--', axis='y')

plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_02_variance_shares.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7c: Sub-period summary table ─────────────────────────────────────────────
# Annualized volatilities (%) averaged within each sub-period.
# Compare directionally against CLMX (2022) Figure 2.

subperiods = {
    '2010–2014': ('2010-01', '2014-12'),
    '2015–2019': ('2015-01', '2019-12'),
    '2020–2021 (COVID)': ('2020-01', '2021-12'),
    '2022–2024': ('2022-01', '2024-12'),
    'Full period': (STUDY_START[:7], STUDY_END[:7]),
}

rows = []
for label, (start, end) in subperiods.items():
    sub = monthly_results.loc[start:end, ['MKT', 'IND', 'FIRM']]
    if sub.empty:
        continue
    # Average monthly variance → annualize → sqrt
    avg_annual_var = sub.mean() * 12
    avg_annual_vol = np.sqrt(avg_annual_var) * 100
    total = avg_annual_var.sum()
    rows.append({
        'Period': label,
        'MKT Vol (%)': f"{avg_annual_vol['MKT']:.1f}",
        'IND Vol (%)': f"{avg_annual_vol['IND']:.1f}",
        'FIRM Vol (%)': f"{avg_annual_vol['FIRM']:.1f}",
        'FIRM Share': f"{avg_annual_var['FIRM']/total:.1%}",
        'Months': len(sub),
    })

summary_df = pd.DataFrame(rows)
print('Sub-period Summary')
print('=' * 75)
print(summary_df.to_string(index=False))
print()
print('CLMX (2022) directional benchmarks for the post-2001 period:')
print('  VW market vol ≈ 18% average (vs. ≈12% in 1962–1997 CLMX period)')
print('  VW industry vol ≈ 14% average (vs. ≈9% in 1962–1997 CLMX period)')
print('  FIRM share of total variance generally LOWER post-2001 than 1962–1997')
print('  (market volatility rose more than firm-specific volatility post-crisis)')

---

## Section 8 — Directional Comparison to CLMX (2022)

We do not attempt exact numerical replication of the 2022 NBER update — our universe (S&P 500 via yfinance) is structurally different from the CRSP universe used in that paper. The comparison is **directional and qualitative**.

### What to look for

From CLMX (2022), the expected pattern for 2010–2024:

| Feature | CLMX (2022) Prediction for Post-2001 | Our Approximate Range |
|---|---|---|
| MKT volatility average | ~18% annualized (VW) | Check Section 7c |
| IND volatility average | ~14% annualized (VW) | Check Section 7c |
| FIRM volatility average | ~18–22% annualized (VW) | Check Section 7c |
| FIRM share of total | Lower than 1962–1997 (≈50%) | Check Section 7c |
| Crisis months | MKT share spikes sharply | March 2020 result above |
| Tech bubble period | FIRM share elevated | Not in our window |

**Note:** Our estimates will likely show lower MKT and FIRM volatility than CLMX because:
- S&P 500 large-cap stocks are less volatile than the full CRSP universe
- Survivorship bias in our universe removes historically distressed companies
- The market index (VW S&P 500) is less volatile than the broader CRSP-VW index

A successful directional replication means the **pattern** is consistent with CLMX, not the level.

In [ ]:
# ── 8a: Directional consistency check ────────────────────────────────────────

# CLMX (2022) qualitative predictions for a post-2001 sample:
#   1. MKT component should spike during GFC (2008–2009) and COVID (March 2020)
#   2. FIRM variance share should be meaningfully lower during crisis months
#   3. FIRM variance share should be relatively higher in low-volatility expansion periods
#   4. No obvious secular upward trend in FIRM share (the pre-2001 trend did not persist)

# Check 1: Crisis month MKT share vs. calm month MKT share
crisis_months = ['2008-10', '2008-11', '2009-03', '2020-03', '2020-04']
calm_months   = ['2017-06', '2017-07', '2018-07', '2019-06', '2019-07']

def month_shares(df, months):
    sel = df.loc[[m for m in months if m in df.index]]
    if sel.empty:
        return None
    total = sel[['MKT', 'IND', 'FIRM']].sum(axis=1)
    return (sel[['MKT', 'IND', 'FIRM']].div(total, axis=0)).mean()

monthly_idx = monthly_results.copy()
monthly_idx.index = monthly_idx.index.strftime('%Y-%m')

crisis_shares = month_shares(monthly_idx, crisis_months)
calm_shares   = month_shares(monthly_idx, calm_months)

print('Directional consistency check:')
print('═' * 50)
if crisis_shares is not None:
    print('Average variance shares — crisis months:')
    for k, v in crisis_shares.items():
        print(f'  {k}: {v:.1%}')
if calm_shares is not None:
    print('\nAverage variance shares — calm months:')
    for k, v in calm_shares.items():
        print(f'  {k}: {v:.1%}')

print()
print('Expected: MKT share higher in crisis months, FIRM share higher in calm months.')
print('This is the most basic directional check against CLMX (2022) Figure 4 behavior.')

In [ ]:
# ── 8b: Secular trend test ────────────────────────────────────────────────────
# CLMX (2022) finding: no persistent secular increase in FIRM share post-2001.
# We test whether there is a significant linear trend in the annual FIRM share
# in our study window. A flat or inconsistent trend is consistent with CLMX (2022).

from scipy import stats

# Annual FIRM share series
firm_share_annual = annual['FIRM_share'].dropna()

if len(firm_share_annual) >= 5:
    years = np.arange(len(firm_share_annual))
    slope, intercept, r_value, p_value, std_err = stats.linregress(years, firm_share_annual)
    print('Linear trend in annual FIRM variance share:')
    print(f'  Slope: {slope:.4f} per year ({slope*100:.2f} percentage points/year)')
    print(f'  R²:    {r_value**2:.3f}')
    print(f'  p-val: {p_value:.3f}')
    print()
    if p_value < 0.05:
        direction = 'increasing' if slope > 0 else 'decreasing'
        print(f'  → Statistically significant {direction} trend (p < 0.05)')
        print(f'    This deserves further investigation under H1.')
    else:
        print(f'  → No statistically significant secular trend (p = {p_value:.2f})')
        print(f'    Consistent with CLMX (2022): the 1962–1997 secular trend did not persist.')
    print()
    print('Caution: This is a preliminary trend test on S&P 500 data.')
    print('It is exploratory. Formal H1 testing requires Phase 2 and Phase 5.')
else:
    print('Insufficient annual data points for trend test.')

---

## Section 9 — Documented Deviations and Limitations

Every meaningful difference between our implementation and the original CLMX study is documented here. These are not excuses — they are preconditions for interpreting any comparison honestly.

### Deviation Table

| Dimension | CLMX (2001 / 2022) | This Replication | Expected Effect |
|---|---|---|---|
| **Universe** | Full CRSP (NYSE, AMEX, NASDAQ, ~3,000–8,000 stocks) | S&P 500 current members (~500 stocks, survivorship-biased) | Lower FIRM volatility levels; understated dispersion; no micro-cap or distressed-company dynamics |
| **Data source** | CRSP (institutional, point-in-time) | yfinance (free, current S&P 500 only) | Survivorship bias; possible gaps for delisted/replaced companies |
| **Survivorship** | Point-in-time CRSP constituents | Current S&P 500 members projected backward | Companies removed from index for poor performance are absent; upward bias in historical "quality" |
| **Market weights** | CRSP daily market cap (price × shrout) | Price × current shares outstanding (static approximation) | Small distortion in weights; larger distortion for companies with significant buybacks or share issuances |
| **Return series** | CRSP total return (dividends + capital gains) | yfinance adjusted close (includes splits + dividends approximately) | Generally comparable; yfinance adjustment quality varies by ticker |
| **Study period** | 1962–2021 (2022 update) | 2010–2024 (prototype) | No comparison to pre-2001 secular rise; only post-crisis period available |
| **Industry classification** | FF49 from CRSP SIC codes (CRSP-assigned) | FF49 from SEC EDGAR SIC codes (EDGAR-filed) | Minor differences possible where EDGAR SIC differs from CRSP SIC |
| **Missing data** | CRSP-standard exclusions (exchange code, share code) | Any return-missing day drops the stock for that month | More aggressive exclusion; may drop thinly traded periods more often |
| **Min observation threshold** | Not documented in CLMX | 10 trading days per month (our design choice) | Documented assumption, not CLMX requirement |

### Survivorship Bias Assessment

This notebook does not resolve survivorship bias — it documents its presence. The Phase 1 notebook (`01_data_validation.ipynb`) contains the formal bias diagnostic. Key implication for this notebook: historical FIRM variance estimates for 2010–2024 are likely **understated** relative to a full-universe panel, because distressed companies and eventual delistees — which tend to have higher idiosyncratic volatility — are absent from our backward-looking S&P 500 sample.

### A Successful Replication Under These Constraints

Given the above deviations, a directionally successful replication means:

1. **Crisis months** show elevated MKT share and depressed FIRM share — consistent with all stocks falling together
2. **Expansion months** show MKT share declining and FIRM share recovering — consistent with idiosyncratic differentiation in calm markets
3. **No obvious secular upward trend in FIRM share** over 2010–2024 — consistent with CLMX (2022) finding that the pre-2001 trend did not persist
4. **Annualized MKT and IND volatilities are lower than CLMX's CRSP-based estimates** by approximately 20–40% — consistent with the large-cap bias of the S&P 500 versus the full CRSP universe

An unsuccessful replication would show:

1. MKT component not spiking during known crisis periods
2. The three components failing to add up approximately to the VW-average individual-stock variance
3. Extreme outlier months with no obvious economic explanation
4. The FIRM component consistently near zero (suggesting the industry matching is incorrect)

If outcomes consistent with the unsuccessful case appear, revisit the industry classification pipeline (Section 3) and the weight calculation (Section 5) before assuming a structural finding.

In [ ]:
# ── 9a: Reconciliation check ─────────────────────────────────────────────────
# Verify that MKT + IND + FIRM ≈ VW-average individual stock monthly variance.
# A large and persistent gap would indicate a computational error.

# For each month, compute VW-average individual stock variance directly
# (Σ_i w_i * Σ_d r_i,d²) and compare to sum of components.

RECON_CACHE = DATA_DIR / 'clmx_reconciliation.csv'

if not RECON_CACHE.exists():
    recon_rows = []
    for dt, row in monthly_results.iterrows():
        year, month = dt.year, dt.month
        mask = (
            (daily_returns.index.year == year) &
            (daily_returns.index.month == month)
        )
        R = daily_returns[mask]
        valid_cols = R.columns[R.notna().all()]
        R = R[valid_cols]
        W = compute_monthly_weights(approx_mktcap, year, month)
        W = W.reindex(valid_cols).fillna(0)
        total_w = W.sum()
        if total_w == 0 or R.empty:
            continue
        W = W / total_w
        vw_total = (R**2).sum().mul(W).sum()
        comp_sum = row['MKT'] + row['IND'] + row['FIRM']
        recon_rows.append({'date': dt, 'VW_total': vw_total, 'comp_sum': comp_sum})

    recon_df = pd.DataFrame(recon_rows).set_index('date')
    recon_df.to_csv(RECON_CACHE)
else:
    recon_df = pd.read_csv(RECON_CACHE, index_col=0, parse_dates=True)

recon_df['diff'] = recon_df['VW_total'] - recon_df['comp_sum']
recon_df['diff_pct'] = (recon_df['diff'] / recon_df['VW_total']).abs()

print('Reconciliation check: VW-total vs. MKT + IND + FIRM')
print(f'Median absolute difference: {recon_df["diff_pct"].median():.2%}')
print(f'Max absolute difference:    {recon_df["diff_pct"].max():.2%}')
print(f'Months with > 5% gap:       {(recon_df["diff_pct"] > 0.05).sum()}')
print()
print('If median difference is < 2%, the decomposition is internally consistent.')
print('Differences arise from cross-product terms (covariance between market/industry/firm')
print('components) that the additive decomposition does not fully capture.')

---

## Replication Checklist

Before considering this replication validated, verify each item:

- [ ] Synthetic example (Section 2) produces sensible MKT/IND/FIRM decomposition
- [ ] FF49 crosswalk loaded and industry assignment coverage > 90% of S&P 500 tickers
- [ ] SIC codes retrieved for > 90% of tickers from EDGAR
- [ ] Daily price data available for the full study window
- [ ] March 2020 shows elevated MKT share (crisis month check)
- [ ] Reconciliation: median VW-total vs. MKT+IND+FIRM difference < 2%
- [ ] Sub-period table shows crisis months with higher MKT share than calm months
- [ ] Annual FIRM share shows no obvious secular upward trend (consistent with CLMX 2022)
- [ ] Annualized MKT/IND/FIRM volatilities are plausible for a large-cap universe

A replication that passes these checks is directionally consistent with the literature and adequate to serve as the methodological foundation for Project Parallax's Phase 2 and Phase 5 analysis.

---

**Next notebook:** `03_dispersion_analysis.ipynb` — Layer 1 and Layer 2 of the analytical framework: cross-sectional dispersion and within-sector correlation structure (H1a).

**Do not proceed to H1 testing.** The replication notebook establishes methodology. Hypothesis testing begins after Phase 1 gates are cleared.